In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [49]:
 pip install -q tensorflow-model-optimization

In [50]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score
from tensorflow_model_optimization.python.core.keras.compat import keras
import tempfile
import zipfile
import os
import tensorflow_model_optimization as tfmot

In [4]:
batch_size = 64
img_height = 224
img_width = 224

data_dir = "/content/drive/MyDrive/TinyML/Datasets/Tomatoes Diseases/"

In [51]:
test_ds = keras.utils.image_dataset_from_directory(
    data_dir + "Test",
    validation_split=0.2,
    subset="training",
    seed=48,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=True,
)

Found 3198 files belonging to 10 classes.
Using 2559 files for training.


In [52]:
train_ds = keras.utils.image_dataset_from_directory(
    data_dir + "Train",
    validation_split=0.2,
    subset="training",
    seed=48,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=True,
)

Found 12860 files belonging to 10 classes.
Using 10288 files for training.


In [53]:
class_names = test_ds.class_names
class_names

['Bacterial_spot',
 'Early_blight',
 'Late_blight',
 'Leaf_Mold',
 'Septoria_leaf_spot',
 'Spider_mite_Two_spotted_sm',
 'Target_Spot',
 'Yellow_Leaf__Curl_Virus',
 'healthy',
 'mosaic_virus']

In [54]:
test_ds = test_ds.take(16)
train_ds = test_ds.take(125)

In [55]:
AUTOTUNE = tf.data.AUTOTUNE

test_nds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)
train_nds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
AlexNet = keras.models.load_model("/content/drive/MyDrive/TinyML/SaveModels/AlexNet.h5")


AlexNet.compile(
  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
  optimizer=keras.optimizers.Adam(learning_rate=1e-5),
  metrics=["accuracy"]
)


In [10]:
model_name = "drive/MyDrive/TinyML/SaveModels/DenseNet.h5"

In [11]:
def get_gzipped_model_size(file, x):
  # Returns size of gzipped model, in bytes.
  zipped_file = "drive/MyDrive/TinyML/SaveModels/" + x
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(file)

  return os.path.getsize(zipped_file) / 1024

In [12]:
print("DenseNet  size: ", get_gzipped_model_size(model_name, "DenseNet.zip"), ' KB')

DenseNet  size:  25571.80078125  KB


In [13]:
pruned_densenet = "drive/MyDrive/TinyML/SaveModels/pruned_DenseNet.h5"

In [14]:
print("Pruned DenseNet size: ", get_gzipped_model_size(pruned_densenet, "pruned_DenseNet.zip"), ' KB')

Pruned DenseNet size:  5585.5322265625  KB


In [15]:
quant_densenet = keras.models.load_model("drive/MyDrive/TinyML/SaveModels/pruned_DenseNet.h5")
quant_densenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
quant_densenet.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 densenet121 (Functional)    (None, 7, 7, 1024)        7037504   
                                                                 
 global_average_pooling2d (  (None, 1024)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dropout (Dropout)           (None, 1024)              0         
                                                                 
 dense (Dense)               (None, 10)                10250     
                                                                 
Total params: 7047754 (26.89 MB)
Trainable params: 6964106 (26.57 MB)
Non-trainable params: 83648 (326.75 KB)
_________________

In [47]:
def representative_data_gen():
  for image, label in train_nds.take(5):
    for img  in image:
      image = np.array(img, dtype=np.float32, ndmin=4)
      yield [image]

In [56]:
converter = tf.lite.TFLiteConverter.from_keras_model(quant_densenet)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_data_gen

#converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.SELECT_TF_OPS]
quant_mobilenet_tflite = converter.convert()

/usr/local/lib/python3.10/dist-packages/tensorflow/lite/python/convert.py:983: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [57]:
tflite_name = "drive/MyDrive/TinyML/SaveModels/ptq_DenseNet.tflite"

with open(tflite_name, "wb") as f:
  f.write(quant_mobilenet_tflite)


print("PTQ TFLite DenseNet size:", get_gzipped_model_size(tflite_name, "ptq_DenseNet.zip"), ' KB')

PTQ TFLite DenseNet size: 1969.40234375  KB


In [58]:
def eval_model(test_ds):
  interpreter = tf.lite.Interpreter(tflite_name)
  interpreter.allocate_tensors()

  input_index = interpreter.get_input_details()[0]["index"]
  output_index = interpreter.get_output_details()[0]["index"]

  prediction_digits = []
  vrai = []
  i = 0
  nbre = 0
  num_correct = 0

  for images, labels in test_ds:

    for img, lab in zip(images, labels):
      if i%100 == 0:
        print(f"Evaluated on {i} results so far.")
      i += 1

      img = np.expand_dims(img, axis=0).astype(np.float32)
      #img = (np.expand_dims(img, axis=0) * 255).astype(np.int8)
      interpreter.set_tensor(input_index, img)

      interpreter.invoke()

      output = interpreter.get_tensor(output_index)
      digit = np.argmax(output)
      prediction_digits.append(digit)


      vrai.append(lab)

      if digit == lab:
        num_correct += 1

  prediction_digits = np.array(prediction_digits)

  return num_correct, prediction_digits, vrai

In [61]:
rescale = keras.layers.Rescaling(1./127.5, offset=-1)

test_nds = test_nds.map(lambda x, y: (rescale(x), y))

In [62]:
n, predict, lab = eval_model(test_nds)

Evaluated on 0 results so far.
Evaluated on 100 results so far.
Evaluated on 200 results so far.
Evaluated on 300 results so far.
Evaluated on 400 results so far.
Evaluated on 500 results so far.
Evaluated on 600 results so far.
Evaluated on 700 results so far.
Evaluated on 800 results so far.
Evaluated on 900 results so far.
Evaluated on 1000 results so far.


In [63]:
print(f"{n}/{1024}\n")

190/1024

